In [0]:
library(dataiku)
library(stats)
library(parallel)
library(dplyr)
library(FNN)
library(cluster)
library(ggplot2)
library(rpart)
library(caret)

In [0]:
# Recipe inputs
hurdle_components_path <- dkuManagedFolderPath("5NPBmWH1")

counterfactual_test_data <- dkuReadDataset("counterfactual_test_data", samplingMethod="head", nbRows=100000)

# read all models and functions as a list

# List all .rds files in the folder
rds_files <- list.files(hurdle_components_path, pattern = "\\.rds$", full.names = TRUE)

# Read all .rds files into a list
models_n_functions_list <- lapply(rds_files, readRDS)

# Print the names of the loaded objects
names(models_n_functions_list) <- basename(rds_files)

# Display the list
print(models_n_functions_list)

In [0]:
# get unique municipality observations
mun_properties  <- counterfactual_test_data %>%
    distinct(Mun_Code,
             blue_ss_frac,
             blue_ls_frac,
             red_ls_frac,
             orange_ls_frac,
             yellow_ss_frac,
             red_ss_frac,
             orange_ss_frac,
             yellow_ls_frac,
             roof_strong_wall_strong,
             roof_strong_wall_light,
             roof_strong_wall_salv,
             roof_light_wall_strong,
             roof_light_wall_light,
             roof_light_wall_salv,
             roof_salv_wall_strong,
             roof_salv_wall_light,
             roof_salv_wall_salv,
             island_groups,
             .keep_all = FALSE)

In [0]:
head(mun_properties)

In [0]:
# Creating a function that generates a counterfactual dataset

#' @title counterfactual_gen
#' @description Function takes argguments of df, tc, matches and returns a
#' counterfactual dataframe to be used by the hurdle function
#' @param df counterfactual_test_data
#' @param tc tropical cyclone name
#' @param matches municipality matches NOT NEEDED
#' @return counterfactual_data_list return

counterfactual_gen  <- function(df, tc, storm_surge, landslide){

    # get unique municipality codes
    mun_code  <- unique(df$Mun_Code)

    # filter df by tc and get hazard characheristics
    counterfactual_data  <- df %>%
        filter(typhoon == tc) %>% # keep the minimum distance from the filter
        mutate(track_min_dist = min(track_min_dist, na.rm = TRUE),
              rain_total = rain_total[which.min(track_min_dist)],
              wind_max = wind_max[which.min(track_min_dist)],
              wind_blue_ss = wind_max * storm_surge,
              wind_yellow_ss = wind_max * storm_surge,
              wind_orange_ss = wind_max * storm_surge,
              wind_red_ss = wind_max * storm_surge,
              rain_blue_ss = rain_total * landslide,
              rain_yellow_ss = rain_total * landslide,
              rain_orange_ss = rain_total * landslide,
              rain_red_ss = rain_total * landslide
              ) %>%
        select(-typhoon)

    # which municipalities are not in the filtered data?
    missing_mun  <- setdiff(mun_code, counterfactual_data$Mun_Code)

    # debugging
    #cat("number of missing municipalities:", sep = " ", length(missing_mun))

    # Check if there are any missing municipalities
    if (length(missing_mun) > 0) {
        # Get the characteristics of the missing mun codes
        #remaining_mun <- df %>%
        #    filter(Mun_Code %in% missing_mun) %>%
        #    select(-typhoon, -rain_total, -wind_max, -track_min_dist)

        # Assign the hazard characteristics from the counterfactual data
        remaining_mun <- df %>%
            filter(Mun_Code %in% missing_mun) %>% # after filtering Mun_Code has duplicates how do we remove duplicates?
            distinct(Mun_Code, .keep_all = TRUE) %>%  # Keeps the first occurrence of each Mun_Code
            mutate(rain_total = unique(counterfactual_data$rain_total),
                   wind_max = unique(counterfactual_data$wind_max),
                   wind_blue_ss = wind_max * storm_surge,
                   wind_yellow_ss = wind_max * storm_surge,
                   wind_orange_ss = wind_max * storm_surge,
                   wind_red_ss = wind_max * storm_surge,
                   rain_blue_ss = rain_total * landslide,
                   rain_yellow_ss = rain_total * landslide,
                   rain_orange_ss = rain_total * landslide,
                   rain_red_ss = rain_total * landslide,
                   damage_perc = 0, # set damage variable to zero or "Damage_below_10"
                   damage_binary = 0,
                   damage_binary_2 = "Damage_below_10"
                  ) %>%
        select(-typhoon)
        # debugging
        cat("number of columns in remaining_mun", sep = " ", ncol(remaining_mun))

        cat("\n number of columns in counterfactual data", sep = " ", ncol(counterfactual_data))

        # Add the remaining municipalities back into the counterfactual data
        counterfactual_data <- rbind(counterfactual_data, remaining_mun)
    }

    # df should have all the 1478 municipalities

    return(counterfactual_data) # returns a dataframe (maybe list for more experiments)
}

In [0]:
melor_2015  <- counterfactual_gen(df = counterfactual_test_data,
                                  tc = "melor2015", 
                                  storm_surge = 0, 
                                  landslide =0)

head(melor_2015)

In [0]:
colnames

In [0]:
# Predict for the counterfactual dataset
# Counterfactual Testing on Typhoon Melor 2015

# extracting the hurdle function from the list of models and functions
hurdle_function  <- models_n_functions_list$hurdle_function

# hurdle fuctions requires:
# @param df the dataframe
# @param base models as a list
# @param high impact models as a list

base_models  <- list(models_n_functions_list$base_clas_full_model,
                     models_n_functions_list$base_rain_model,
                     models_n_functions_list$base_reg_model,
                     models_n_functions_list$base_wind_model
                    )

# makes sure the list has correct names
names(base_models)  <- c("base_clas_full_model","base_rain_model", "base_reg_model", "base_wind_model")


trunc_models  <- list(models_n_functions_list$trunc_rain_model,
                      models_n_functions_list$trunc_reg_model,
                      models_n_functions_list$trunc_wind_model
                    )

# makes sure the list has correct names
names(trunc_models)  <- c("trunc_rain_model","trunc_reg_model", "trunc_wind_model")


counterfactual_hurdle_preds  <- hurdle_function(df = melor_2015,
                                               scm_models_base = base_models,
                                               scm_models_high = trunc_models,
                                               threshold = 0.35 # threshold in train/test models is 0.35
                                               )

In [0]:
# append the results to the counterfactual dataset
melor_2015  <- melor_2015 %>%
    mutate(damage_preds = counterfactual_hurdle_preds)

In [0]:
colnames(melor_2015)

In [0]:
ggplot(melor_2015, aes(x = factor(island_groups), y = damage_preds)) +
  geom_boxplot()

In [0]:
# Recipe outputs
fixed_sec_hazards_counterfactuals <- dkuManagedFolderPath("Zcih9bxs")